In [7]:
import pandas as pd
import numpy as np

In [16]:
import pandas as pd

# -----------------------------------
# 1. Load data
# -----------------------------------
Index_A = pd.read_csv("data/index_A.csv")   # must contain "index" and "provider name"
Heat    = pd.read_csv("data/Heat.csv")          # must contain "Processes"
A       = pd.read_csv("data/A.csv")         # numeric matrix (685 × 685)

# -----------------------------------
# 2. Use the existing "index" column as row ID
# -----------------------------------
if "index" not in Index_A.columns:
    raise KeyError("Index_A must contain an 'index' column representing row numbers.")

if "provider name" not in Index_A.columns:
    raise KeyError("Index_A must contain 'provider name'.")

# Ensure index values are integers
Index_A["index"] = Index_A["index"].astype(int)

# -----------------------------------
# 3. Match Heat["Processes"] to Index_A["provider name"]
#    (keep ALL duplicates)
# -----------------------------------
matched = Heat[["Processes"]].merge(
    Index_A[["index", "provider name"]],
    left_on="Processes",
    right_on="provider name",
    how="left"
)

matched_indices = matched["index"].dropna().astype(int).tolist()

if not matched_indices:
    print("No matches found.")
    result_df = pd.DataFrame(columns=["index", "provider name"])
    result_df.to_csv("matched_processes.csv", index=False)
    raise SystemExit

# -----------------------------------
# 4. For each matched row index, find non-zero columns in A
# -----------------------------------
nonzero_target_indices = []

for r_idx in matched_indices:
    row = A.iloc[r_idx]   # use row index directly

    nz_cols = row[(row != 0) & row.notna()].index

    for col_name in nz_cols:

        # Case 1: column name is numeric (an index into Index_A)
        try:
            col_idx = int(col_name)
            nonzero_target_indices.append(col_idx)
            continue
        except:
            pass

        # Case 2: column name is a provider name
        mask = Index_A["provider name"] == col_name
        if mask.any():
            nonzero_target_indices.extend(Index_A.loc[mask, "index"].tolist())

# -----------------------------------
# Remove invalid values and keep duplicates
# -----------------------------------
nonzero_target_indices = [
    i for i in nonzero_target_indices
    if i in Index_A["index"].values
]

# -----------------------------------
# 5. Retrieve process names for those indices
# -----------------------------------
result_df = (
    Index_A[Index_A["index"].isin(nonzero_target_indices)][["index", "provider name"]]
    .sort_values("index")
)

print(result_df)

# -----------------------------------
# 6. Save to CSV
# -----------------------------------
result_df.to_csv("matched_processes.csv", index=False)
print("Saved to matched_processes.csv")

Empty DataFrame
Columns: [index, provider name]
Index: []
Saved to matched_processes.csv


In [17]:
import pandas as pd

# -----------------------------
# 1. Load data
# -----------------------------
Index_A = pd.read_csv("data/index_A.csv")   # must contain "index" and "provider name"
Heat    = pd.read_csv("data/Heat.csv")          # must contain "Processes"
A       = pd.read_csv("data/A.csv")         # numeric matrix (685 × 685)

# Basic checks
if "index" not in Index_A.columns:
    raise KeyError("Index_A must contain an 'index' column.")
if "provider name" not in Index_A.columns:
    raise KeyError("Index_A must contain a 'provider name' column.")
if "Processes" not in Heat.columns:
    raise KeyError("Heat must contain a 'Processes' column.")

# Make sure "index" is integer
Index_A["index"] = Index_A["index"].astype(int)

# -----------------------------
# 2. Clean names and match Heat → Index_A
# -----------------------------
Index_A["provider_clean"] = (
    Index_A["provider name"].astype(str).str.strip().str.lower()
)
Heat["proc_clean"] = (
    Heat["Processes"].astype(str).str.strip().str.lower()
)

matched = Heat[["proc_clean"]].merge(
    Index_A[["index", "provider_clean"]],
    left_on="proc_clean",
    right_on="provider_clean",
    how="left"
)

matched_indices = matched["index"].dropna().astype(int).tolist()

if not matched_indices:
    print("No matches between Heat['Processes'] and Index_A['provider name'].")
    result_df = pd.DataFrame(columns=["index", "provider name"])
    result_df.to_csv("matched_processes.csv", index=False)
    raise SystemExit

# -----------------------------
# Helper: convert A column name → numeric index
# -----------------------------
def colname_to_idx(col_name):
    """
    Convert A's column names to integer indices:
      '0'    -> 0
      '0.1'  -> 1
      '0.2'  -> 2
      ...
      '0.684'-> 684
    Anything else (e.g. '1.0') returns None.
    """
    s = str(col_name)
    if s == "0":
        return 0
    if s.startswith("0."):
        try:
            return int(s.split(".", 1)[1])
        except ValueError:
            return None
    return None  # ignore columns like '1.0'

# -----------------------------
# 3. For each matched row in A, find non-zero columns
# -----------------------------
nonzero_target_indices = []

n_rows_A = len(A)

for r_idx in matched_indices:
    # Skip indices that are outside A's row range (Index_A has 686 rows, A has 685)
    if r_idx < 0 or r_idx >= n_rows_A:
        continue

    row = A.iloc[r_idx]
    nz_cols = row[(row != 0) & row.notna()].index

    for col_name in nz_cols:
        col_idx = colname_to_idx(col_name)
        if col_idx is not None:
            nonzero_target_indices.append(col_idx)

# If nothing found, stop here
if not nonzero_target_indices:
    print("No non-zero entries found in A for the matched rows (after parsing column names).")
    result_df = pd.DataFrame(columns=["index", "provider name"])
    result_df.to_csv("matched_processes.csv", index=False)
    raise SystemExit

# -----------------------------
# 4. Go back to Index_A: map indices → provider names
# -----------------------------
nonzero_target_indices_series = pd.Series(nonzero_target_indices, name="index")
result_df = nonzero_target_indices_series.to_frame().merge(
    Index_A[["index", "provider name"]],
    on="index",
    how="left"
)

# Keep rows where we successfully found a provider name
result_df = result_df.dropna(subset=["provider name"]).sort_values("index")

# Optional: if you want to remove exact duplicates (same index & provider name):
# result_df = result_df.drop_duplicates()

print(result_df)

# -----------------------------
# 5. Save to CSV
# -----------------------------
result_df.to_csv("matched_processes.csv", index=False)
print("Saved to matched_processes.csv")


    index                                      provider name
0      78              Disposal, Multi-material Food Bottles
1      98  Disposal, Multi-material Food Packaging Film (...
2     100  Low-tech Sorting, Collected LDPE Food Packagin...
14    142   High-tech Sorting, Collected PP Other Food Rigid
3     146                      Disposal, PP Other Food Rigid
22    156                                  HDPE Food Bottles
4     172       High-tech Sorting, Collected PP Food Bottles
15    173                     Disposal, PET Drinking Bottles
20    212                                 Fossil Electricity
11    236       Transport, single unit truck, diesel powered
5     284  Natural gas, combusted in industrial boiler, a...
6     302       Lignite coal, combusted in industrial boiler
9     309  Wood waste, unspecified, combusted in industri...
12    331  Liquefied petroleum gas, combusted in industri...
16    338    Anthracite coal, combusted in industrial boiler
13    354               

In [18]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# 1. Read the data
# ---------------------------------------------------
Index_A = pd.read_csv("data/index_A.csv")   # has columns: "index", "provider name", ...
Heat    = pd.read_csv("data/Heat.csv")          # has column: "Processes"
A       = pd.read_csv("data/A.csv")         # 685 × 686

# Make sure the index column is integer
Index_A["index"] = Index_A["index"].astype(int)

# ---------------------------------------------------
# 2. From Heat → Index_A: get the row indices of A
#    (match Heat['Processes'] with Index_A['provider name'])
# ---------------------------------------------------
matched = Heat[["Processes"]].merge(
    Index_A[["index", "provider name"]],
    left_on="Processes",
    right_on="provider name",
    how="left"
)

row_indices = matched["index"].dropna().astype(int).tolist()

if not row_indices:
    print("No matches between Heat['Processes'] and Index_A['provider name'].")
    result_df = pd.DataFrame(columns=["index", "provider name"])
    result_df.to_csv("matched_processes.csv", index=False)
    raise SystemExit

# ---------------------------------------------------
# 3. For those row indices in A, find non-zero columns
#    (column index = position, starting from 0, left to right)
# ---------------------------------------------------
n_rows, n_cols = A.shape

column_indices = []  # keep all, including duplicates if they occur

for r in row_indices:
    # Skip indices that are outside A's row range
    if r < 0 or r >= n_rows:
        continue

    row = A.iloc[r].to_numpy()
    nz_cols = np.nonzero(row)[0]     # these are 0-based column positions
    column_indices.extend(nz_cols.tolist())

if not column_indices:
    print("No non-zero entries found in A for the matched rows.")
    result_df = pd.DataFrame(columns=["index", "provider name"])
    result_df.to_csv("matched_processes.csv", index=False)
    raise SystemExit

# ---------------------------------------------------
# 4. For those column indices, look them up in Index_A["index"]
#    and get the corresponding provider names
# ---------------------------------------------------
col_idx_df = pd.DataFrame({"index": column_indices})

result_df = col_idx_df.merge(
    Index_A[["index", "provider name"]],
    on="index",
    how="left"
).sort_values("index")

# (Optional) if you want unique providers only, uncomment:
# result_df = result_df.drop_duplicates()

print(result_df)

# ---------------------------------------------------
# 5. Save to CSV
# ---------------------------------------------------
result_df.to_csv("matched_processes.csv", index=False)
print("Saved to matched_processes.csv")

    index                                      provider name
0      79                     Disposal, PET Non-food Bottles
1      99                         Disposal, PET Food Bottles
2     101                     Disposal, PET Other Food Rigid
14    143   Low-tech Sorting, Collected PET Drinking Bottles
3     147                 Disposal, PET Other Non-food Rigid
22    157             Reclaiming, Sorted PS Other Food Rigid
4     173                     Disposal, PET Drinking Bottles
15    174  High-tech Sorting, Collected PET Drinking Bottles
20    213              Winter wheat straw, ground and stored
11    237  Fuels, burned at bleached kraft market pulp mi...
5     285                                  Open Burning, PET
6     303          Enzyme, Alpha-amylase, Novozyme Liquozyme
9     310  Wastewater treatment; Anaerobic/anoxic/oxic, L...
12    332  Bituminous coal, combusted in industrial boile...
16    339            Reclaiming, Sorted PET Drinking Bottles
13    355  Fuels, burned

In [19]:
row_indices

[284, 302, 309, 331, 338, 577, 578, 658]

In [20]:
matched

,Processes,index,provider name
0,"Natural gas, combusted in industrial boiler, a...",284,"Natural gas, combusted in industrial boiler, a..."
1,"Lignite coal, combusted in industrial boiler",302,"Lignite coal, combusted in industrial boiler"
2,"Wood waste, unspecified, combusted in industri...",309,"Wood waste, unspecified, combusted in industri..."
3,"Liquefied petroleum gas, combusted in industri...",331,"Liquefied petroleum gas, combusted in industri..."
4,"Anthracite coal, combusted in industrial boiler",338,"Anthracite coal, combusted in industrial boiler"
5,"Wood fuel, hardwood, generated at lumber mill,...",577,"Wood fuel, hardwood, generated at lumber mill,..."
6,"Wood fuel, hardwood, purchased, combusted in i...",578,"Wood fuel, hardwood, purchased, combusted in i..."
7,"Natural gas, combusted in industrial boiler, a...",658,"Natural gas, combusted in industrial boiler, a..."


In [21]:
import pandas as pd
import numpy as np

# 1. Read data
Index_A = pd.read_csv("data/index_A.csv")   # has columns: "index", "provider name", ...
Heat    = pd.read_csv("data/Heat.csv")          # has column: "Processes"
A       = pd.read_csv("data/A.csv")         # numeric matrix

# Make sure "index" is integer ID
Index_A["index"] = Index_A["index"].astype(int)

# 2. Match Heat['Processes'] with Index_A['provider name']
#    and store the *value* in the "index" column (row IDs for A)
matched = Heat[["Processes"]].merge(
    Index_A[["index", "provider name"]],
    left_on="Processes",
    right_on="provider name",
    how="left"
)

row_ids_for_A = matched["index"].dropna().astype(int).tolist()
# (These are e.g. [284, 302, 309, 331, 338, 577, 578, 658] on your data)

# 3. For those row IDs in A, find every non-zero value and
#    store the 0-based column positions
n_rows, n_cols = A.shape
col_ids = []

for r in row_ids_for_A:
    if r < 0 or r >= n_rows:
        # skip if the id is outside A's row range
        continue

    row = A.iloc[r].to_numpy()
    nz_cols = np.nonzero(row)[0]      # 0-based column indices
    col_ids.extend(nz_cols.tolist())

if not col_ids:
    print("No non-zero entries found in A for the matched rows.")
    result_df = pd.DataFrame(columns=["index", "provider name"])
    result_df.to_csv("matched_processes.csv", index=False)
else:
    # 4. For those stored column numbers, search in Index_A['index']
    #    and get the corresponding 'provider name'
    col_idx_df = pd.DataFrame({"index": col_ids})

    result_df = (
        col_idx_df
        .merge(Index_A[["index", "provider name"]], on="index", how="left")
        .sort_values("index")
    )

    # If you want unique providers only, uncomment:
    # result_df = result_df.drop_duplicates()

    print(result_df)

    # 5. Save to CSV
    result_df.to_csv("matched_processes.csv", index=False)
    print("Saved to matched_processes.csv")


    index                                      provider name
0      79                     Disposal, PET Non-food Bottles
1      99                         Disposal, PET Food Bottles
2     101                     Disposal, PET Other Food Rigid
14    143   Low-tech Sorting, Collected PET Drinking Bottles
3     147                 Disposal, PET Other Non-food Rigid
22    157             Reclaiming, Sorted PS Other Food Rigid
4     173                     Disposal, PET Drinking Bottles
15    174  High-tech Sorting, Collected PET Drinking Bottles
20    213              Winter wheat straw, ground and stored
11    237  Fuels, burned at bleached kraft market pulp mi...
5     285                                  Open Burning, PET
6     303          Enzyme, Alpha-amylase, Novozyme Liquozyme
9     310  Wastewater treatment; Anaerobic/anoxic/oxic, L...
12    332  Bituminous coal, combusted in industrial boile...
16    339            Reclaiming, Sorted PET Drinking Bottles
13    355  Fuels, burned

In [22]:
row_ids_for_A

[284, 302, 309, 331, 338, 577, 578, 658]

In [23]:
matched

,Processes,index,provider name
0,"Natural gas, combusted in industrial boiler, a...",284,"Natural gas, combusted in industrial boiler, a..."
1,"Lignite coal, combusted in industrial boiler",302,"Lignite coal, combusted in industrial boiler"
2,"Wood waste, unspecified, combusted in industri...",309,"Wood waste, unspecified, combusted in industri..."
3,"Liquefied petroleum gas, combusted in industri...",331,"Liquefied petroleum gas, combusted in industri..."
4,"Anthracite coal, combusted in industrial boiler",338,"Anthracite coal, combusted in industrial boiler"
5,"Wood fuel, hardwood, generated at lumber mill,...",577,"Wood fuel, hardwood, generated at lumber mill,..."
6,"Wood fuel, hardwood, purchased, combusted in i...",578,"Wood fuel, hardwood, purchased, combusted in i..."
7,"Natural gas, combusted in industrial boiler, a...",658,"Natural gas, combusted in industrial boiler, a..."


In [24]:
import pandas as pd
import numpy as np

# 1. Read data
Index_A = pd.read_csv("data/index_A.csv")   # has: "index", "provider name", ...
Heat    = pd.read_csv("data/Heat.csv")          # has: "Processes"
A       = pd.read_csv("data/A.csv")         # numeric matrix

# Ensure "index" is integer ID
Index_A["index"] = Index_A["index"].astype(int)

# 2. Match Heat['Processes'] to Index_A['provider name']
#    and get the *values* in the "index" column (row IDs in A)
matched = Heat[["Processes"]].merge(
    Index_A[["index", "provider name"]],
    left_on="Processes",
    right_on="provider name",
    how="left"
)

row_ids_for_A = matched["index"].dropna().astype(int).tolist()

if not row_ids_for_A:
    print("No matches between Heat['Processes'] and Index_A['provider name'].")
    out = pd.DataFrame(columns=[
        "heat_index", "heat_name", "user_index", "user_name", "value_in_A"
    ])
    out.to_csv("heat_users.csv", index=False)
    raise SystemExit

n_rows, n_cols = A.shape

# 3. For each heat row in A, find non-zero columns and record the mapping
records = []

for r in row_ids_for_A:
    if r < 0 or r >= n_rows:
        # skip indices outside A's row range
        continue

    # heat process name for this row id (take first match if duplicated)
    heat_name = Index_A.loc[Index_A["index"] == r, "provider name"]
    heat_name = heat_name.iloc[0] if not heat_name.empty else None

    row = A.iloc[r].to_numpy()
    nz_cols = np.nonzero(row)[0]          # 0-based column indices

    for c in nz_cols:
        value = row[c]
        records.append({
            "heat_index": r,
            "heat_name": heat_name,
            "user_index": c,              # column index (consumer process)
            "value_in_A": value           # optional: strength of connection
        })

if not records:
    print("No non-zero entries found in A for the matched heat rows.")
    out = pd.DataFrame(columns=[
        "heat_index", "heat_name", "user_index", "user_name", "value_in_A"
    ])
    out.to_csv("heat_users.csv", index=False)
    raise SystemExit

# 4. For each user_index (column index), look up provider name in Index_A["index"]
result_df = pd.DataFrame(records)

result_df = result_df.merge(
    Index_A[["index", "provider name"]],
    left_on="user_index",
    right_on="index",
    how="left"
)

result_df = (
    result_df
    .rename(columns={"provider name": "user_name"})
    .drop(columns=["index"])
    .sort_values(["heat_index", "user_index"])
    .reset_index(drop=True)
)

print(result_df)

# 5. Save to CSV
result_df.to_csv("heat_users.csv", index=False)
print("Saved to heat_users.csv")

    heat_index                                          heat_name  user_index  \
0          284  Natural gas, combusted in industrial boiler, a...          79   
1          284  Natural gas, combusted in industrial boiler, a...          99   
2          284  Natural gas, combusted in industrial boiler, a...         101   
3          284  Natural gas, combusted in industrial boiler, a...         147   
4          284  Natural gas, combusted in industrial boiler, a...         173   
5          284  Natural gas, combusted in industrial boiler, a...         285   
6          302       Lignite coal, combusted in industrial boiler         303   
7          302       Lignite coal, combusted in industrial boiler         402   
8          302       Lignite coal, combusted in industrial boiler         632   
9          309  Wood waste, unspecified, combusted in industri...         310   
10         309  Wood waste, unspecified, combusted in industri...         560   
11         331  Liquefied pe

In [25]:
import pandas as pd
import numpy as np

# ---------------------------
# 1. Load data
# ---------------------------
Index_A = pd.read_csv("data/index_A.csv")   # has "index", "provider name", ...
Heat    = pd.read_csv("data/Heat.csv")          # has "Processes"
A       = pd.read_csv("data/A.csv")         # 685 x 686 matrix

# ensure "index" is integer ID
Index_A["index"] = Index_A["index"].astype(int)

# ---------------------------
# 2. Map Heat processes -> row IDs in A
#    (use the value in Index_A["index"])
# ---------------------------
matched = Heat[["Processes"]].merge(
    Index_A[["index", "provider name"]],
    left_on="Processes",
    right_on="provider name",
    how="left"
)

heat_row_ids = matched["index"].dropna().astype(int).tolist()

if not heat_row_ids:
    print("No matches between Heat['Processes'] and Index_A['provider name'].")
    out = pd.DataFrame(columns=["heat_index", "heat_name",
                                "user_index", "user_name", "value_in_A"])
    out.to_csv("heat_users.csv", index=False)
    raise SystemExit

n_rows, n_cols = A.shape

# ---------------------------
# Helper: column label -> process index
# ---------------------------
def col_label_to_index(col_label):
    """
    Your A columns look like: '1.0', '0', '0.1', '0.2', ..., '0.684'.
    Only labels starting with '0' encode a process index:
        '0'     -> 0
        '0.78'  -> 78
        '0.284' -> 284
    '1.0' is ignored (no process index).
    """
    s = str(col_label)
    if s == "0":
        return 0
    if s.startswith("0."):
        try:
            return int(s.split(".", 1)[1])
        except ValueError:
            return None
    return None  # ignore '1.0' and anything else

# ---------------------------
# 3. For each heat row in A, find non-zero columns
#    and map them to processes
# ---------------------------
records = []

for r in heat_row_ids:
    if r < 0 or r >= n_rows:
        # skip invalid row ids
        continue

    # name of this heat flow (input row)
    heat_name_series = Index_A.loc[Index_A["index"] == r, "provider name"]
    heat_name = heat_name_series.iloc[0] if not heat_name_series.empty else None

    row = A.iloc[r].to_numpy()
    nz_cols = np.nonzero(row)[0]   # positions of non-zero entries

    for col_pos in nz_cols:
        col_label = A.columns[col_pos]
        user_idx = col_label_to_index(col_label)
        if user_idx is None:
            # column doesn't correspond to a process index (e.g. '1.0'), skip
            continue

        value = row[col_pos]

        records.append({
            "heat_index": r,        # ID of the heat flow (row)
            "heat_name": heat_name,
            "user_index": user_idx, # ID of the process using this heat
            "value_in_A": value
        })

if not records:
    print("No non-zero entries found in A for the matched heat rows.")
    out = pd.DataFrame(columns=["heat_index", "heat_name",
                                "user_index", "user_name", "value_in_A"])
    out.to_csv("heat_users.csv", index=False)
    raise SystemExit

# ---------------------------
# 4. Map user_index -> provider name (process name)
# ---------------------------
result_df = pd.DataFrame(records)

result_df = result_df.merge(
    Index_A[["index", "provider name"]],
    left_on="user_index",
    right_on="index",
    how="left"
)

result_df = (
    result_df
    .rename(columns={"provider name": "user_name"})
    .drop(columns=["index"])
    .sort_values(["heat_index", "user_index"])
    .reset_index(drop=True)
)

print(result_df.head(30))  # preview

# ---------------------------
# 5. Save to CSV
# ---------------------------
result_df.to_csv("heat_users.csv", index=False)
print("Saved to heat_users.csv")


    heat_index                                          heat_name  user_index  \
0          284  Natural gas, combusted in industrial boiler, a...          78   
1          284  Natural gas, combusted in industrial boiler, a...          98   
2          284  Natural gas, combusted in industrial boiler, a...         100   
3          284  Natural gas, combusted in industrial boiler, a...         146   
4          284  Natural gas, combusted in industrial boiler, a...         172   
5          284  Natural gas, combusted in industrial boiler, a...         284   
6          302       Lignite coal, combusted in industrial boiler         302   
7          302       Lignite coal, combusted in industrial boiler         401   
8          302       Lignite coal, combusted in industrial boiler         631   
9          309  Wood waste, unspecified, combusted in industri...         309   
10         309  Wood waste, unspecified, combusted in industri...         559   
11         331  Liquefied pe

In [1]:
import pandas as pd
import numpy as np

# 1. Load data
Index_A = pd.read_csv("data/index_A.csv")   # has: "index", "provider name", ...
Heat    = pd.read_csv("data/Heat.csv")          # has: "Processes"
A       = pd.read_csv("data/A.csv")         # rows = flows, cols = processes

Index_A["index"] = Index_A["index"].astype(int)

# 2. Match Heat["Processes"] → Index_A["provider name"] → row IDs in A
matched = Heat[["Processes"]].merge(
    Index_A[["index", "provider name"]],
    left_on="Processes",
    right_on="provider name",
    how="left"
)

heat_row_ids = matched["index"].dropna().astype(int).tolist()

if not heat_row_ids:
    print("No matches between Heat['Processes'] and Index_A['provider name'].")
    out = pd.DataFrame(columns=[
        "heat_index", "heat_name", "user_index", "user_name", "value_in_A"
    ])
    out.to_csv("heat_users.csv", index=False)
    raise SystemExit

n_rows, n_cols = A.shape

# 3. Helper: map A column label → process index from Index_A["index"]
def col_label_to_index(col_label):
    s = str(col_label)
    if s == "0":
        return 0
    if s.startswith("0."):
        try:
            return int(s.split(".", 1)[1])
        except ValueError:
            return None
    # ignore "1.0" and anything else
    return None

# 4. For each heat row in A, find processes that use that heat
records = []

for r in heat_row_ids:
    if r < 0 or r >= n_rows:
        continue

    heat_name_series = Index_A.loc[Index_A["index"] == r, "provider name"]
    heat_name = heat_name_series.iloc[0] if not heat_name_series.empty else None

    row = A.iloc[r].to_numpy()

    # Option A: any non-zero = "uses heat"
    mask = row != 0

    # If you want "inputs only" and you know the sign convention,
    # you can use row > 0 or row < 0 instead.

    nz_positions = np.nonzero(mask)[0]

    for pos in nz_positions:
        col_label = A.columns[pos]
        user_idx = col_label_to_index(col_label)

        # skip columns that don't map to a process index
        if user_idx is None:
            continue

        # skip the heat process itself (diag)
        if user_idx == r:
            continue

        value = row[pos]

        records.append({
            "heat_index": r,
            "heat_name": heat_name,
            "user_index": user_idx,
            "value_in_A": value
        })

if not records:
    print("No non-zero entries found in A for the matched heat rows.")
    out = pd.DataFrame(columns=[
        "heat_index", "heat_name", "user_index", "user_name", "value_in_A"
    ])
    out.to_csv("heat_users.csv", index=False)
    raise SystemExit

# 5. Map user_index → provider name (process name)
result_df = pd.DataFrame(records)

result_df = result_df.merge(
    Index_A[["index", "provider name"]],
    left_on="user_index",
    right_on="index",
    how="left"
)

result_df = (
    result_df
    .rename(columns={"provider name": "user_name"})
    .drop(columns=["index"])
    .sort_values(["heat_index", "user_index"])
    .reset_index(drop=True)
)

print(result_df.head(30))

# 6. Save to CSV
result_df.to_csv("heat_users.csv", index=False)
print("Saved to heat_users.csv")


    heat_index                                          heat_name  user_index  \
0          284  Natural gas, combusted in industrial boiler, a...          78   
1          284  Natural gas, combusted in industrial boiler, a...          98   
2          284  Natural gas, combusted in industrial boiler, a...         100   
3          284  Natural gas, combusted in industrial boiler, a...         146   
4          284  Natural gas, combusted in industrial boiler, a...         172   
5          302       Lignite coal, combusted in industrial boiler         401   
6          302       Lignite coal, combusted in industrial boiler         631   
7          309  Wood waste, unspecified, combusted in industri...         559   
8          331  Liquefied petroleum gas, combusted in industri...         236   
9          331  Liquefied petroleum gas, combusted in industri...         354   
10         338    Anthracite coal, combusted in industrial boiler         142   
11         338    Anthracite